In [ ]:
import pandas as pd
frame_mfcc = pd.read_pickle('**********.pkl')
frame_mfcc

In [ ]:
frame_mfcc = frame_mfcc[[
    'mfcc0', 'mfcc1', 'mfcc2', 'mfcc3', 'mfcc4', 'mfcc5', 'mfcc6', 'mfcc7',
       'mfcc8', 'mfcc9', 'mfcc10', 'mfcc11', 'mfcc12', 'mfcc13', 'mfcc14',
       'mfcc15', 'mfcc16', 'mfcc17', 'mfcc18', 'mfcc19', 'mfcc20', 'mfcc21',
       'mfcc22', 'mfcc23', 'mfcc24', 'mfcc25', 'mfcc26', 'mfcc27', 'mfcc28',
       'mfcc29',

   'shimmerLocaldB_sma3nz_amean',
       'shimmerLocaldB_sma3nz_stddevNorm', 'loudness_sma3_amean',
      'loudness_sma3_stddevNorm', 'HNRdBACF_sma3nz_amean',
       'HNRdBACF_sma3nz_stddevNorm',

    'loudnessPeaksPerSec',
        'MeanVoicedSegmentLengthSec', 'StddevVoicedSegmentLengthSec',
       'MeanUnvoicedSegmentLength', 'StddevUnvoicedSegmentLength',
        'VoicedSegmentsPerSec',

    'F0semitoneFrom27.5Hz_sma3nz_amean',
    'F0semitoneFrom27.5Hz_sma3nz_stddevNorm', 'jitterLocal_sma3nz_amean',
    'jitterLocal_sma3nz_stddevNorm', 'F1frequency_sma3nz_amean',
    'F1frequency_sma3nz_stddevNorm', 'F2frequency_sma3nz_amean',
    'F2frequency_sma3nz_stddevNorm', 'F3frequency_sma3nz_stddevNorm',
    'F3amplitudeLogRelF0_sma3nz_amean', 'F1bandwidth_sma3nz_amean',
    'F1bandwidth_sma3nz_stddevNorm',

    'label', 'name'
                         ]]

In [ ]:
import os
import json
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import tensorflow as tf
import keras
from keras import layers
from tensorflow.keras.optimizers.legacy import Adam
from itertools import product

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, balanced_accuracy_score, confusion_matrix
)
from scipy.stats import friedmanchisquare





GLOBAL_SEED = 42
OUTPUT_DIR = "./svm_search_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)



# 1. Data Preparation

X = frame_mfcc.iloc[:, :54].to_numpy()
X = np.expand_dims(X, axis=1)          # shape: (N, 1, 54)
y = np.array(frame_mfcc['label'])

print("X shape:", X.shape, "y shape:", y.shape)
print("Class distribution:", np.bincount(y.astype(int)))


In [ ]:
# 2. Model Definition

def locally_enhanced_ffn(inputs, ff_dim, dropout, kernel_sizeDepth,
                         activation1, activation2):
    x = layers.Conv1D(filters=ff_dim, kernel_size=1,
                      activation=activation1)(inputs)
    x = layers.DepthwiseConv1D(kernel_size=kernel_sizeDepth,
                               padding='same')(x)
    x = layers.Activation(activation2)(x)
    x = layers.Dropout(dropout)(x)
    x = layers.Conv1D(filters=inputs.shape[-1], kernel_size=1,
                      activation=activation2)(x)
    x = layers.Add()([x, inputs])
    return x


def transformer_encoder(inputs, head_size, num_heads, ff_dim, dropout,
                        kernel_sizeDepth, activation1, activation2):
    x = layers.MultiHeadAttention(key_dim=head_size, num_heads=num_heads,
                                  dropout=dropout)(inputs, inputs)
    x = layers.Dropout(dropout)(x)
    x = layers.LayerNormalization(epsilon=1e-6)(x)
    res = x + inputs
    x = locally_enhanced_ffn(res, ff_dim, dropout, kernel_sizeDepth,
                             activation1, activation2)
    x = layers.LayerNormalization(epsilon=1e-6)(x)
    return x


def build_model(input_shape, head_size, num_heads, ff_dim,
                num_transformer_blocks, dropout, kernel_sizeDepth,
                activation1, activation2, units):
    inputs = keras.Input(shape=input_shape)
    x = inputs
    for _ in range(num_transformer_blocks):
        x = transformer_encoder(x, head_size, num_heads, ff_dim, dropout,
                                kernel_sizeDepth, activation1, activation2)
    x = layers.Bidirectional(layers.LSTM(units=units,
                                         return_sequences=False))(x)
    x = layers.Dropout(dropout)(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)
    return keras.Model(inputs, outputs)


# 3. Fixed parameters

fixed_params = {
    'head_size': 2,
    'ff_dim': 64,
    'num_transformer_blocks': 2,
    'activation1': 'relu',
    'activation2': 'gelu',
    'max_epochs': 100,
    'learning_rate': 1e-4,
    'batch_size': 32,
    'early_stop_patience': 10,
    'early_stop_min_delta': 1e-3,
    'val_split_for_early_stop': 0.1,   
}

# 4. Search space

search_space = {
    'num_heads':        [2, 4, 6, 8],
    'units':            [32, 64, 128, 256],
    'dropout':          [0.0,],# try 0.0, 0.1, 0.3, 0.5. So 4 times running, to avoid long time waiting or running out of memory
    'kernel_sizeDepth': [1, 3],
}

keys = list(search_space.keys())
combos = list(product(*[search_space[k] for k in keys]))
print(f"Total parameter combinations: {len(combos)}")

In [ ]:


# 5. Fixed 5 fold split 

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=GLOBAL_SEED)
folds = list(skf.split(X, y))
print(f"Number of folds: {len(folds)}")



# 6. Main loop

METRIC = 'auc'   # Options: 'accuracy', 'f1', 'auc', 'balanced_accuracy'

results = {}        
all_metric_records = []  

for ci, combo in enumerate(combos):
    params = dict(zip(keys, combo))
    print(f"\n===== Combo {ci+1}/{len(combos)}: {params} =====")

    fold_scores = []
    fold_records = []

    for fi, (train_idx, test_idx) in enumerate(folds):
        # Fix the random seed for this fold to ensure consistent initialization across all combinations.
        set_seed(GLOBAL_SEED + fi)

        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        X_tr, X_val, y_tr, y_val = train_test_split(
            X_train, y_train,
            test_size=fixed_params['val_split_for_early_stop'],
            stratify=y_train,
            random_state=GLOBAL_SEED + fi,
        )

        scaler = StandardScaler()

        X_tr_rs    = X_tr.reshape(X_tr.shape[0], X_tr.shape[2])
        X_val_rs   = X_val.reshape(X_val.shape[0], X_val.shape[2])
        X_test_rs  = X_test.reshape(X_test.shape[0], X_test.shape[2])

        X_tr_scaled    = scaler.fit_transform(X_tr_rs)
        X_val_scaled   = scaler.transform(X_val_rs)
        X_test_scaled  = scaler.transform(X_test_rs)

        X_tr    = X_tr_scaled.reshape(X_tr.shape[0], X_tr.shape[1], X_tr.shape[2])
        X_val   = X_val_scaled.reshape(X_val.shape[0], X_val.shape[1], X_val.shape[2])
        X_test  = X_test_scaled.reshape(X_test.shape[0], X_test.shape[1], X_test.shape[2])


        tf.keras.backend.clear_session()

        model = build_model(
            input_shape=(X_tr.shape[1], X_tr.shape[2]),
            head_size=fixed_params['head_size'],
            num_heads=params['num_heads'],
            ff_dim=fixed_params['ff_dim'],
            num_transformer_blocks=fixed_params['num_transformer_blocks'],
            dropout=params['dropout'],
            kernel_sizeDepth=params['kernel_sizeDepth'],
            activation1=fixed_params['activation1'],
            activation2=fixed_params['activation2'],
            units=params['units'],
        )

        model.compile(
            loss='binary_crossentropy',
            optimizer=Adam(learning_rate=fixed_params['learning_rate']),
            metrics=[tf.keras.metrics.AUC(name='auc')],
        )


        early_stop = tf.keras.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=fixed_params['early_stop_patience'],
            min_delta=fixed_params['early_stop_min_delta'],
            restore_best_weights=True,
            verbose=0,
        )

        history = model.fit(
            X_tr, y_tr,
            validation_data=(X_val, y_val),
            epochs=fixed_params['max_epochs'],
            batch_size=fixed_params['batch_size'],
            callbacks=[early_stop],
            verbose=2,
        )

        epochs_ran = len(history.history['loss'])

        # prediction
        y_prob = model.predict(X_test, verbose=0).flatten()
        y_pred = (y_prob >= 0.5).astype(int).flatten()
        y_test_flat = np.array(y_test).flatten()

        # results
        acc   = accuracy_score(y_test_flat, y_pred)
        f1    = f1_score(y_test_flat, y_pred)
        prec  = precision_score(y_test_flat, y_pred, zero_division=0)
        rec   = recall_score(y_test_flat, y_pred, zero_division=0)
        auc   = roc_auc_score(y_test_flat, y_prob)
        balacc = balanced_accuracy_score(y_test_flat, y_pred)

        tn, fp, fn, tp = confusion_matrix(y_test_flat, y_pred).ravel()
        sen = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        spe = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        ppv = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        npv = tn / (tn + fn) if (tn + fn) > 0 else 0.0

        if METRIC == 'f1':
            score = f1
        elif METRIC == 'accuracy':
            score = acc
        elif METRIC == 'auc':
            score = auc
        elif METRIC == 'balanced_accuracy':
            score = balacc
        else:
            raise ValueError("Unknown METRIC")

        fold_scores.append(score)
        fold_records.append({
            'fold': fi,
            'epochs_ran': epochs_ran,
            'accuracy': float(acc),
            'f1': float(f1),
            'precision': float(prec),
            'recall': float(rec),
            'auc': float(auc),
            'balanced_accuracy': float(balacc),
            'sensitivity': float(sen),
            'specificity': float(spe),
            'ppv': float(ppv),
            'npv': float(npv),
            'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp),
        })


    fold_scores = np.array(fold_scores)
    results[combo] = fold_scores

    all_metric_records.append({
        'combo_index': ci,
        'params': params,
        'scores': fold_scores.tolist(),
        'mean': float(fold_scores.mean()),
        'std': float(fold_scores.std(ddof=1)),
        'fold_details': fold_records,
    })

    print(f"  {METRIC}: {fold_scores.mean():.4f} ± {fold_scores.std(ddof=1):.4f}")

In [ ]:

# 8. Summary: Read 4 sets of results, concatenate + Friedman test

import os
import pickle
import numpy as np
from scipy.stats import friedmanchisquare

OUTPUT_DIR = "******/Downloads/search_results"
dropout_values = [0.0, 0.1, 0.3, 0.5]

merged_results = {}      # combo -> scores (np.array)
merged_records = []     

for dv in dropout_values:
    f = os.path.join(OUTPUT_DIR, f"results_dropout_{dv}.pkl")
    if not os.path.exists(f):
        print(f"[Warning] Missing {f}, skip.")
        continue
    with open(f, 'rb') as fp:
        data = pickle.load(fp)
    merged_results.update(data['results'])
    merged_records.extend(data['all_metric_records'])

print(f"Total combos merged: {len(merged_results)}")


keys = ['num_heads', 'units', 'dropout', 'kernel_sizeDepth']

summary = []
for combo, scores in merged_results.items():
    params = dict(zip(keys, combo))
    summary.append({
        'params': params,
        'mean': float(np.mean(scores)),
        'std':  float(np.std(scores, ddof=1)),
    })

summary_sorted = sorted(summary, key=lambda x: -x['mean'])

print("\n---- Top 20 by mean ----")
for r in summary_sorted[:20]:
    print(f"{r['mean']:.4f} ± {r['std']:.4f} | {r['params']}")

# ---------- Friedman test ----------
score_matrix = np.array([merged_results[c] for c in merged_results.keys()])
print(f"\nScore matrix shape: {score_matrix.shape}")

stat, p_val = friedmanchisquare(*score_matrix)
print(f"Friedman chi2 = {stat:.4f}, p = {p_val:.6f}")

if p_val > 0.05:
    print("=> No significant differences were detected; parameters can be selected randomly.")
    rng = np.random.RandomState(42)
    chosen = summary_sorted[rng.randint(len(summary_sorted))]
else:
    print("=> There are significant differences; select the parameters that yield the best performance.")
    chosen = summary_sorted[0]

print(f"\n[Chosen] {chosen['params']}")
print(f"Performance: {chosen['mean']:.4f} ± {chosen['std']:.4f}")

# ----------Save the result----------
import json
out_file = os.path.join(OUTPUT_DIR, "merged_summary_sorted.json")
with open(out_file, 'w') as f:
    json.dump({
        'n_combos': len(summary_sorted),
        'summary_sorted': summary_sorted,
        'friedman_chi2': float(stat),
        'friedman_p': float(p_val),
        'chosen': chosen,
    }, f, indent=2, default=str)

print(f"\n[Saved] {out_file}")

In [ ]:

import scikit_posthocs as sp
import numpy as np

score_matrix = np.array([merged_results[c] for c in merged_results.keys()])

# Nemenyi post-hoc test
p_matrix = sp.posthoc_nemenyi_friedman(score_matrix.T)  
print(p_matrix)

In [ ]:
# merge
merged_results = {}
for dv in dropout_values:
    f = os.path.join(OUTPUT_DIR, f"results_dropout_{dv}.pkl")
    if not os.path.exists(f):
        print(f"[Warning] Missing {f}, skip.")
        continue
    with open(f, 'rb') as fp:
        data = pickle.load(fp)
    merged_results.update(data['results'])

keys_list = list(merged_results.keys())
score_matrix = np.array([merged_results[k] for k in keys_list])
n_combos, n_folds = score_matrix.shape
print(f"Score matrix shape: {score_matrix.shape}")

# ---- 1. Friedman ----
stat, p_val = friedmanchisquare(*score_matrix)
print(f"\n[Friedman] chi2 = {stat:.4f}, p = {p_val:.6f}")

# ---- 2. Kendall's W ----
W = stat / (n_folds * (n_combos - 1))
print(f"[Kendall's W] {W:.4f}  (0.1=small, 0.3=medium, 0.5=large)")

# ---- 3. Nemenyi post-hoc test ----
p_matrix = sp.posthoc_nemenyi_friedman(score_matrix.T)
p_matrix_np = p_matrix.values if hasattr(p_matrix, "values") else np.asarray(p_matrix)
print(f"\n[Nemenyi] p_matrix shape: {p_matrix_np.shape}")

means = score_matrix.mean(axis=1)
top1_idx = int(np.argmax(means))
print(f"Top1: {dict(zip(keys, keys_list[top1_idx]))}, mean={means[top1_idx]:.4f}")

equivalent_group = [i for i in range(n_combos) if p_matrix_np[top1_idx, i] > 0.05]
print(f"[Nemenyi] Number of combinations not significantly different from the top-ranked one: {len(equivalent_group)} / {n_combos}")

print("\nCombinations with no significant difference from the top-ranked (Top 1) option")
for i in equivalent_group:
    print(f"  mean={means[i]:.4f} ± {score_matrix[i].std(ddof=1):.4f} | "
          f"{dict(zip(keys, keys_list[i]))}")




In [ ]:
# Identify the combinations that do not differ significantly from Top1
equivalent_group = [i for i in range(n_combos) if p_matrix_np[top1_idx, i] > 0.05]
print(f"Equivalent group size: {len(equivalent_group)} / {n_combos}")



In [ ]:
target = (2, 64, 0.5, 1)   # (num_heads, units, dropout, kernel_sizeDepth)

# Confirm that this combo is in keys_list.
if target not in keys_list:
    print(f"[Error] {target} not in keys_list; please check the tuple order or parameter values.")
else:
    target_idx = keys_list.index(target)
    means = score_matrix.mean(axis=1)

    print(f"combo: {dict(zip(keys, target))}")
    print(f"index: {target_idx}")
    print(f"mean = {means[target_idx]:.4f}, std = {score_matrix[target_idx].std(ddof=1):.4f}")

    if target_idx in equivalent_group:
        print("In the 121 equivalent group (with no significant difference from Top1)")
    else:
        print("Not in the 121 equivalence group (significantly different from Top1)")

